In [ ]:
# ============================================================
# mise_a_jour.py
# ============================================================
# Ce script :
#   1. Lit data_finale.csv existant
#   2. Détecte les mois manquants
#   3. Télécharge et ajoute les nouvelles données
#   4. Sauvegarde le fichier mis à jour
#
# Usage : python mise_a_jour.py
# ============================================================

In [ ]:
# Bibliothèques

import camelot
import requests
import pandas as pd
import re
import time
import itertools
import datetime
import os

In [ ]:
# =============== CONFIGURATION ===============
BASE_URL  = "https://www.acea.auto/files/"
HEADERS   = {"User-Agent": "Mozilla/5.0"}
DATA_RAW  = "data/raw"
DATA_PROC = "data/processed"

MOIS_NOMS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

GROUPES_CIBLES = {
    "Volkswagen Group": "Volkswagen Group",
    "Stellantis":       "Stellantis",
    "Renault Group":    "Renault Group",
    "Toyota Group":     "Toyota Group"
}

os.makedirs(DATA_RAW,  exist_ok=True)
os.makedirs(DATA_PROC, exist_ok=True)

In [ ]:
# =============== FONCTIONS (identiques à build_data_finale.py) ===============

def construire_url(annee, mois_num):
    nom = MOIS_NOMS[mois_num - 1]
    if (annee > 2023) or (annee == 2023 and mois_num >= 10):
        if annee == 2024 and mois_num == 8:
            return f"{BASE_URL}Press_release_car_registrations_August-2024.pdf"
        return f"{BASE_URL}Press_release_car_registrations_{nom}_{annee}.pdf"
    mois_pub  = mois_num + 1 if mois_num < 12 else 1
    annee_pub = annee if mois_num < 12 else annee + 1
    yymm      = f"{str(annee)[2:]}{mois_num:02d}"
    for jour, prefixe, suffixe in itertools.product(
        range(14, 25), ["PRPC", "PCPR"], ["_FINAL", "-FINAL"]
    ):
        date_pub = f"{annee_pub}{mois_pub:02d}{jour:02d}"
        filename = f"{date_pub}_{prefixe}_{yymm}{suffixe}.pdf"
        url      = BASE_URL + filename
        r = requests.head(url, headers=HEADERS, timeout=10, allow_redirects=False)
        if r.status_code == 200 and yymm in filename:
            return url
    return None


def identifier_table_eu_efta_uk(tables):
    groupes    = list(GROUPES_CIBLES.keys())
    candidates = []
    for i, table in enumerate(tables):
        clean_matches = partial_matches = 0
        for _, row in table.df.iterrows():
            col0 = str(row.iloc[0]).strip()
            if not col0 or col0 == 'nan':
                continue
            for groupe in groupes:
                if col0 == groupe or col0.startswith(groupe + '\n'):
                    clean_matches += 1
                    break
                elif groupe in col0:
                    partial_matches += 1
                    break
        if clean_matches + partial_matches >= 2:
            total_num = sum(
                int(re.sub(r'[^\d]', '', str(v)))
                for v in table.df.values.flatten()
                if re.sub(r'[^\d]', '', str(v))
            )
            candidates.append((i, table, clean_matches, total_num))
    if not candidates:
        return None, None
    candidates.sort(key=lambda x: (x[2], x[3]), reverse=True)
    idx, table, _, _ = candidates[0]
    return idx, table


def nettoyer_brand(s):
    return re.sub(r'\d+$', '', s).strip()

def nettoyer_valeur(s):
    clean = re.sub(r'[^\d]', '', str(s))
    return int(clean) if clean else None

def extraire_groupe_depuis_col0(col0):
    parts = [p.strip() for p in col0.split('\n') if p.strip()]
    for part in reversed(parts):
        if part in GROUPES_CIBLES:
            return GROUPES_CIBLES[part]
    return None

def parse_inline(names, values, current_group, results):
    for name, val in zip(names, values):
        name = name.strip()
        if not name:
            continue
        if name in GROUPES_CIBLES:
            current_group = GROUPES_CIBLES[name]
            continue
        if name.lower().endswith("group") and name not in GROUPES_CIBLES:
            current_group = None
            continue
        if current_group is None:
            continue
        if re.match(r'^[\d.,+\-\s]+$', name):
            continue
        brand = nettoyer_brand(name)
        if not brand or brand.lower() == 'total':
            continue
        registrations = nettoyer_valeur(val)
        if registrations:
            results.append({
                "group":         current_group,
                "brand":         brand,
                "registrations": registrations
            })
    return current_group

def parse_tableau(table_df, col_valeurs=3):
    results       = []
    current_group = None
    rows          = list(table_df.iterrows())
    for i, (_, row) in enumerate(rows):
        nb_cols = len(row)
        col0    = str(row.iloc[0]).strip()
        col_val = str(row.iloc[col_valeurs]).strip() if nb_cols > col_valeurs else ''
        if not col0 or col0 == 'nan':
            continue
        mots_cles = [
            'DECEMBER','JANUARY','FEBRUARY','MARCH','APRIL','MAY',
            'JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER',
            'UNITS','SHARE','JAN-','JUL-','APR-'
        ]
        if any(h in col0.upper() for h in mots_cles):
            continue
        names  = [n.strip() for n in col0.split('\n') if n.strip()]
        values = [v.strip() for v in col_val.split('\n') if v.strip()]
        first  = names[0]
        if first in GROUPES_CIBLES:
            current_group = GROUPES_CIBLES[first]
            if len(names) > 1:
                brand_names  = names[1:]
                brand_values = values[1:] if len(values) > 1 else []
                if not brand_values and i + 1 < len(rows):
                    _, next_row = rows[i + 1]
                    next_col0   = str(next_row.iloc[0]).strip()
                    next_col_v  = str(next_row.iloc[col_valeurs]).strip() if len(next_row) > col_valeurs else ''
                    if not next_col0 or next_col0 == 'nan':
                        brand_values = [v.strip() for v in next_col_v.split('\n') if v.strip()]
                current_group = parse_inline(brand_names, brand_values, current_group, results)
        else:
            groupe_detecte = extraire_groupe_depuis_col0(col0)
            if groupe_detecte:
                current_group = groupe_detecte
                brand_names = [
                    n for n in names
                    if not re.match(r'^[\d.,+\-\s]+$', n)
                    and n not in GROUPES_CIBLES
                    and not (n.lower().endswith("group") and n not in GROUPES_CIBLES)
                ]
                current_group = parse_inline(brand_names, values, current_group, results)
            elif current_group is not None:
                current_group = parse_inline(names, values, current_group, results)
    return pd.DataFrame(results)


def telecharger_pdf(url, pdf_path):
    if os.path.exists(pdf_path):
        return True
    response = requests.get(url, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        return False
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    return True

def extraire_donnees(pdf_path, col_valeurs=3):
    try:
        tables = camelot.read_pdf(pdf_path, pages="all", flavor="lattice")
    except Exception as e:
        return None, f"camelot: {e}"
    idx, table = identifier_table_eu_efta_uk(tables)
    if table is None:
        return None, "table non détectée"
    df = parse_tableau(table.df, col_valeurs=col_valeurs)
    if df.empty:
        return None, "dataframe vide"
    return df, None

In [ ]:
# ════════════════════════════════════════════════════════════
# MISE À JOUR
# ════════════════════════════════════════════════════════════

def mise_a_jour():
    path_csv = f"{DATA_PROC}/data_finale.csv"

    print("=" * 60)
    print("MISE À JOUR MENSUELLE")
    print("=" * 60)

    # 1. Chargement de data_finale existant
    data_finale = pd.read_csv(path_csv)
    print(f"data_finale chargé : {data_finale.shape[0]} lignes")

    # 2. Identification du dernier mois disponible
    df_mensuel    = data_finale[data_finale["month_num"] > 0]
    dernier_annee = int(df_mensuel["year"].max())
    dernier_mois  = int(df_mensuel[df_mensuel["year"] == dernier_annee]["month_num"].max())
    print(f"==== Dernières données  : {MOIS_NOMS[dernier_mois-1]} {dernier_annee} ====")

    # 3. Calcul des mois manquants jusqu'à (mois courant - 1)
    aujourd_hui = datetime.date.today()
    mois_cible  = aujourd_hui.month - 1 or 12
    annee_cible = aujourd_hui.year if aujourd_hui.month > 1 else aujourd_hui.year - 1

    mois_manquants = []
    annee, mois = dernier_annee, dernier_mois + 1
    if mois > 12:
        mois  = 1
        annee += 1
    while (annee < annee_cible) or (annee == annee_cible and mois <= mois_cible):
        mois_manquants.append((annee, mois))
        mois += 1
        if mois > 12:
            mois  = 1
            annee += 1

    if not mois_manquants:
        print("==== Déjà à jour - aucun mois manquant ====")
        return

    print(f"{len(mois_manquants)} mois à récupérer : "
          f"{MOIS_NOMS[mois_manquants[0][1]-1]} {mois_manquants[0][0]}"
          f" → {MOIS_NOMS[mois_manquants[-1][1]-1]} {mois_manquants[-1][0]}")

    # 4. Récupération des mois manquants
    nouveaux = []
    erreurs  = []

    for annee_m, mois_num_m in mois_manquants:
        mois_nom_m = MOIS_NOMS[mois_num_m - 1]
        print(f"\n{mois_nom_m} {annee_m}")

        url = construire_url(annee_m, mois_num_m)
        if not url:
            print(f"----URL introuvable")
            erreurs.append((mois_nom_m, annee_m, "URL non trouvée"))
            continue

        print(f"{url.split('/')[-1]}")
        pdf_path = f"{DATA_RAW}/acea_{mois_nom_m}_{annee_m}.pdf"

        if not telecharger_pdf(url, pdf_path):
            print(f"==== Téléchargement échoué ====")
            erreurs.append((mois_nom_m, annee_m, "téléchargement échoué"))
            continue

        df, err = extraire_donnees(pdf_path, col_valeurs=3)
        if err:
            print(f"{err}")
            erreurs.append((mois_nom_m, annee_m, err))
            continue

        df["month"]     = mois_nom_m
        df["month_num"] = mois_num_m
        df["year"]      = annee_m
        nouveaux.append(df)
        print(f"{len(df)} lignes — {df['group'].nunique()} groupes")
        time.sleep(0.5)

    # 5. Ajout et sauvegarde
    if not nouveaux:
        print("\n---- Aucune nouvelle donnée récupérée")
        return

    cols = ["year", "month_num", "month", "group", "brand", "registrations"]
    df_nouveaux = pd.concat(nouveaux, ignore_index=True)[cols]
    data_finale = pd.concat([data_finale, df_nouveaux], ignore_index=True)
    data_finale = data_finale.sort_values(
        ["year", "month_num", "group", "brand"]
    ).reset_index(drop=True)

    data_finale.to_csv(path_csv, index=False)

    print(f"\n==== data_finale mis à jour ====")
    print(f"   Total lignes : {data_finale.shape[0]} (+{len(df_nouveaux)} nouvelles)")
    print(f"   Fichier      : {path_csv}")

    if erreurs:
        print(f"\n-----{len(erreurs)} mois non récupérés :")
        for e in erreurs:
            print(f"   {e[0]} {e[1]} → {e[2]}")

In [ ]:
# ════════════════════════════════════════════════════════════
# LANCEMENT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    mise_a_jour()